# 05 — Selección, evaluación final y Model Registry

La selección de champion y challenger se hace con CV. Solo esas dos opciones se evalúan una vez en el holdout. El RMSE del holdout no se usa para volver a elegir el modelo.

## Límite de recursos observado

Las 30 configuraciones de Random Forest completaron su CV, pero el entorno no pudo serializar el bosque final con bootstrap sin límite. Para cerrar la evaluación de forma reproducible, se valida por CV una variante de recursos limitados (`max_samples=4096`): cada árbol se ajusta con 4.096 filas extraídas de toda la partición de entrenamiento. La desviación se registra en MLflow; la selección final se basa en el CV de esta variante y el RMSE del holdout se usa solo para reportar desempeño.

In [1]:
from pathlib import Path
import ast
import gc
import os
import sys
import time
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.entities import ViewType
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_validate
from src.business_metrics import calibrate_pairwise_interval, coelution_proxy_metrics
from src.datos import load_snapshot
from src.experiments import EXPERIMENT_NAME, MODEL_PIP_REQUIREMENTS, SCORING, WORKFLOW_VERSION, _git_commit, configure_tracking
from src.pipeline import make_default_models, make_search_spaces, make_model_pipeline, chromatographic_condition_groups, model_inputs, valid_smiles_mask
from src.splits import create_or_load_holdout, stratified_compound_sample

frame, metadata = load_snapshot()
valid = valid_smiles_mask(frame)
all_data = frame.loc[valid & frame["inchikey"].notna()].reset_index(drop=True)
train, test, split_metadata = create_or_load_holdout(all_data, source_sha256=metadata["sha256"])
selection_sample, sample_metadata = stratified_compound_sample(train, max_compounds=12_000, random_state=42)
X_oof, y_oof, groups_oof = model_inputs(selection_sample)
train_rows, holdout_rows = len(train), len(test)
del frame, all_data, valid, selection_sample
gc.collect()
X_train, y_train, groups_train = model_inputs(train)
X_test, y_test, groups_test = model_inputs(test)
del train, test
gc.collect()
final_metadata = {**metadata, "source_rows": metadata["rows"], "rows": train_rows, "training_sample_sha256": sample_metadata["sha256"], "sampling_method": sample_metadata["method"], "sampling_seed": sample_metadata["random_state"]}
tuning_table = pd.read_csv(PROJECT_ROOT / "docs" / "tuning.csv")
configure_tracking()
client = MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
model_name = "qsrr-predict-modelo"
best_per_family = tuning_table.sort_values("cv_rmse_mean").groupby("family", as_index=False).first()
best_per_family["tuned_best_cv_rmse_mean"] = best_per_family["cv_rmse_mean"]
best_per_family["tuned_best_cv_rmse_std"] = best_per_family["cv_rmse_std"]
def parse_parameter(value, candidates):
    if value == "None":
        return None
    template = next((candidate for candidate in candidates if candidate is not None), None) if isinstance(candidates, list) else None
    if isinstance(template, tuple):
        return ast.literal_eval(value)
    if template is not None:
        return type(template)(value)
    return float(value)

rf_row = best_per_family.loc[best_per_family["family"].eq("random_forest")].iloc[0]
rf_search_space = make_search_spaces()["random_forest"]
rf_best_parameters = {
    name: parse_parameter(client.get_run(rf_row["run_id"]).data.params[name], candidates)
    for name, candidates in rf_search_space.items()
}
rf_resource_rows = 4_096
existing_rf_resource_runs = client.search_runs(
    [experiment.experiment_id],
    filter_string=(
        "tags.phase = 'resource_validation' AND tags.model_family = 'random_forest' "
        f"AND tags.training_sample_sha256 = '{sample_metadata['sha256']}'"
    ),
    run_view_type=ViewType.ALL,
    max_results=100,
)
finished_rf_resource_runs = [run for run in existing_rf_resource_runs if run.info.status == "FINISHED"]
if finished_rf_resource_runs:
    rf_resource_run = max(finished_rf_resource_runs, key=lambda item: item.info.start_time)
    rf_resource_cv_rmse = rf_resource_run.data.metrics["resource_adjusted_cv_rmse_mean"]
    rf_resource_cv_std = rf_resource_run.data.metrics["resource_adjusted_cv_rmse_std"]
else:
    from src.pipeline import clear_feature_caches
    clear_feature_caches()
    rf_cv_model = make_model_pipeline(make_default_models()["random_forest"])
    rf_cv_model.set_params(**rf_best_parameters, regressor__max_samples=rf_resource_rows)
    rf_resource_scores = cross_validate(
        rf_cv_model, X_oof, y_oof, groups=groups_oof,
        cv=GroupKFold(n_splits=5), scoring=SCORING, n_jobs=1,
        error_score="raise",
    )
    rf_resource_cv_rmse = float((-rf_resource_scores["test_rmse"]).mean())
    rf_resource_cv_std = float((-rf_resource_scores["test_rmse"]).std(ddof=1))
    with mlflow.start_run(run_name="resource_validation_random_forest"):
        mlflow.set_tags({
            "phase": "resource_validation", "model_family": "random_forest",
            "workflow_version": WORKFLOW_VERSION,
            "git_commit": _git_commit(), "data_sha256": metadata["sha256"],
            "training_sample_sha256": sample_metadata["sha256"],
            "validation": "GroupKFold by inchikey on training sample only",
            "resource_profile": f"max_samples={rf_resource_rows}",
        })
        mlflow.log_param("tuning_best_run_id", rf_row["run_id"])
        mlflow.log_param("training_sample_rows", sample_metadata["rows"])
        mlflow.log_param("final_bootstrap_rows", rf_resource_rows)
        mlflow.log_params(rf_best_parameters)
        mlflow.log_metrics({
            "resource_adjusted_cv_rmse_mean": rf_resource_cv_rmse,
            "resource_adjusted_cv_rmse_std": rf_resource_cv_std,
        })
best_per_family.loc[best_per_family["family"].eq("random_forest"), "cv_rmse_mean"] = rf_resource_cv_rmse
best_per_family.loc[best_per_family["family"].eq("random_forest"), "cv_rmse_std"] = rf_resource_cv_std
finalists = best_per_family.nsmallest(2, "cv_rmse_mean").copy()
display(finalists[["family", "configuration", "cv_rmse_mean", "cv_rmse_std", "run_id"]])
print({"random_forest_resource_adjusted_cv_rmse": rf_resource_cv_rmse, "max_samples": rf_resource_rows})
print({"full_train_rows": len(y_train), "full_grouped_holdout_rows": len(y_test), "selection_sample_compounds": sample_metadata["compounds"], "selection_sample_rows": sample_metadata["rows"]})
trained = {}
final_results = []

,family,configuration,cv_rmse_mean,cv_rmse_std,run_id
0,hist_gradient_boosting,9,2.591868,0.249749,bed738a932bf4200af0654fd3e25fc93
2,random_forest,23,2.593382,0.216498,b2b57d876e3d4a9494d37c4a066388d4


{'random_forest_resource_adjusted_cv_rmse': 2.5933822306253544, 'max_samples': 4096}
{'full_train_rows': 122370, 'full_grouped_holdout_rows': 30216, 'selection_sample_compounds': 12000, 'selection_sample_rows': 19406}


In [2]:
existing_final_runs = client.search_runs(
    [experiment.experiment_id],
    filter_string=(
        f"tags.phase = 'final' AND tags.workflow_version = '{WORKFLOW_VERSION}' "
        f"AND tags.training_sample_sha256 = '{sample_metadata['sha256']}'"
    ),
    run_view_type=ViewType.ALL,
    max_results=100,
)
model_versions = client.search_model_versions(f"name='{model_name}'")
for _, row in finalists.iterrows():
    family = row["family"]
    expected_resource_profile = (
        f"random_forest max_samples={rf_resource_rows}; all training rows are bootstrap population"
        if family == "random_forest" else "full-training refit in Codespace"
    )
    reusable = [
        run for run in existing_final_runs
        if run.info.status == "FINISHED"
        and run.data.tags.get("model_family") == family
        and run.data.tags.get("resource_profile") == expected_resource_profile
    ]
    if reusable:
        run = max(reusable, key=lambda item: item.info.start_time)
        model = mlflow.sklearn.load_model(f"runs:/{run.info.run_id}/model")
        prediction = model.predict(X_test)
        version = next(item for item in model_versions if item.run_id == run.info.run_id)
        metrics = {
            name: run.data.metrics[name]
            for name in (
                "full_train_rmse", "full_train_mae", "holdout_rmse", "holdout_mae",
                "fit_seconds_full_train", "prediction_seconds_full_holdout",
            )
        }
        metrics["selection_sample_cv_rmse_mean"] = float(row["cv_rmse_mean"])
        metrics["selection_sample_cv_rmse_std"] = float(row["cv_rmse_std"])
        trained[family] = {
            "prediction": prediction, "run_id": run.info.run_id,
            "version": str(version.version), **metrics,
        }
        final_results.append({"family": family, "run_id": run.info.run_id, **metrics})
        del model, prediction
        from src.pipeline import clear_feature_caches
        clear_feature_caches()
        gc.collect()
        continue

    search_parameters = make_search_spaces()[family]
    if family == "random_forest":
        best_parameters = rf_best_parameters.copy()
    else:
        logged_parameters = client.get_run(row["run_id"]).data.params
        best_parameters = {
            name: parse_parameter(logged_parameters[name], candidates)
            for name, candidates in search_parameters.items()
        }
    model = make_model_pipeline(make_default_models()[family])
    model.set_params(**best_parameters)
    if family == "random_forest":
        model.set_params(regressor__max_samples=rf_resource_rows)

    fit_start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - fit_start
    train_prediction = model.predict(X_train)
    predict_start = time.perf_counter()
    prediction = model.predict(X_test)
    prediction_seconds = time.perf_counter() - predict_start
    metrics = {
        "full_train_rmse": float(root_mean_squared_error(y_train, train_prediction)),
        "full_train_mae": float(mean_absolute_error(y_train, train_prediction)),
        "holdout_rmse": float(root_mean_squared_error(y_test, prediction)),
        "holdout_mae": float(mean_absolute_error(y_test, prediction)),
        "fit_seconds_full_train": float(fit_seconds),
        "prediction_seconds_full_holdout": float(prediction_seconds),
        "selection_sample_cv_rmse_mean": float(row["cv_rmse_mean"]),
        "selection_sample_cv_rmse_std": float(row["cv_rmse_std"]),
    }
    resource_profile = (
        f"random_forest max_samples={rf_resource_rows}; all training rows are bootstrap population"
        if family == "random_forest" else "full-training refit in Codespace"
    )
    with mlflow.start_run(run_name=f"final_full_train_{family}"):
        mlflow.set_tags({
            "phase": "final", "model_family": family,
            "workflow_version": WORKFLOW_VERSION, "git_commit": _git_commit(),
            "data_tables": metadata["source_tables"],
            "data_snapshot_read_at_utc": metadata["snapshot_read_at_utc"],
            "data_sha256": metadata["sha256"],
            "training_sample_sha256": sample_metadata["sha256"],
            "sampling_method": sample_metadata["method"],
            "sampling_seed": str(sample_metadata["random_state"]),
            "training_scope": "all rows in full grouped train partition",
            "holdout_scope": "all rows in full grouped holdout partition",
            "resource_profile": resource_profile,
            "validation": "selection GroupKFold by inchikey; final full grouped holdout",
            "random_state": "42",
        })
        mlflow.log_param("tuning_run_id", row["run_id"])
        mlflow.log_param("source_rows", final_metadata["source_rows"])
        mlflow.log_param("training_partition_rows", final_metadata["rows"])
        mlflow.log_param("training_sample_rows", sample_metadata["rows"])
        mlflow.log_param("full_training_rows", train_rows)
        mlflow.log_param("full_holdout_rows", holdout_rows)
        if family == "random_forest":
            mlflow.log_param("final_resource_max_samples", rf_resource_rows)
        mlflow.log_params(best_parameters)
        mlflow.log_metrics(metrics)
        signature = infer_signature(X_train.head(5), model.predict(X_train.head(5)))
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature, input_example=X_train.head(5), code_paths=["src"], pip_requirements=MODEL_PIP_REQUIREMENTS)
        final_run_id = mlflow.active_run().info.run_id
    version = mlflow.register_model(f"runs:/{final_run_id}/model", model_name)
    trained[family] = {"prediction": prediction, "run_id": final_run_id, "version": str(version.version), **metrics}
    final_results.append({"family": family, "run_id": final_run_id, **metrics})
    del model, train_prediction, prediction
    from src.pipeline import clear_feature_caches
    clear_feature_caches()
    gc.collect()

final_table = pd.DataFrame(final_results).sort_values("selection_sample_cv_rmse_mean")
display(final_table)
champion_family = final_table.iloc[0]["family"]
challenger_family = final_table.iloc[1]["family"]
client.set_registered_model_alias(model_name, "champion", trained[champion_family]["version"])
challenger_version = next(version.version for version in client.search_model_versions(f"name='{model_name}'") if version.run_id == trained[challenger_family]["run_id"])
client.set_registered_model_alias(model_name, "challenger", str(challenger_version))
print({"champion": champion_family, "challenger": challenger_family})

,family,run_id,full_train_rmse,full_train_mae,holdout_rmse,holdout_mae,fit_seconds_full_train,prediction_seconds_full_holdout,selection_sample_cv_rmse_mean,selection_sample_cv_rmse_std
0,hist_gradient_boosting,04483473d3314b51b7ee40f03ca90c94,2.070066,1.337390,2.328972,1.529281,45.900967,11.087477,2.591868,0.249749
1,random_forest,b49f230a7813494481c464fbce79cef5,2.303174,1.468461,2.506167,1.633446,49.212310,12.858549,2.593382,0.216498


{'champion': 'hist_gradient_boosting', 'challenger': 'random_forest'}


In [3]:
champion = mlflow.sklearn.load_model(f"runs:/{trained[champion_family]['run_id']}/model")
oof_prediction = cross_val_predict(champion, X_oof, y_oof, groups=groups_oof, cv=GroupKFold(n_splits=5), n_jobs=1)
interval = calibrate_pairwise_interval(
    y_oof, oof_prediction,
    chromatographic_condition_groups(X_oof), groups_oof,
    coverage=0.95, random_state=42,
)
business_proxy = coelution_proxy_metrics(
    y_test, trained[champion_family]["prediction"],
    chromatographic_condition_groups(X_test), groups_test, interval,
    random_state=42,
)
for metric_name, metric_value in {**interval, **business_proxy}.items():
    if np.isfinite(float(metric_value)):
        client.log_metric(trained[champion_family]["run_id"], metric_name, float(metric_value))
client.log_param(trained[champion_family]["run_id"], "pairwise_calibration_rows", len(y_oof))
client.log_param(trained[champion_family]["run_id"], "pairwise_calibration_groups", groups_oof.nunique())
client.set_tag(trained[champion_family]["run_id"], "pairwise_calibration_scope", "stratified training sample; grouped 5-fold OOF")
client.set_tag(trained[champion_family]["run_id"], "coelution_labels_are_proxy", "true")
display({"intervalo_95_pairwise": interval, "proxy_sin_etiquetas_de_coelucion": business_proxy})
print("El recall/FNR es solo un proxy de cercanía temporal; no es una tasa real de coelución ni demuestra ahorro de corridas.")

registered_champion = mlflow.sklearn.load_model(f"models:/{model_name}@champion")
reproduced_rmse = float(root_mean_squared_error(y_test, registered_champion.predict(X_test)))
expected_rmse = float(trained[champion_family]["holdout_rmse"])
assert np.isclose(reproduced_rmse, expected_rmse, rtol=0, atol=1e-12)
print({"holdout_rmse_registrado": reproduced_rmse, "reproduccion_exacta": True})

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.workflow_version = '{WORKFLOW_VERSION}'",
    output_format="pandas",
)
rf_runs = client.search_runs(
    [experiment.experiment_id],
    filter_string=(
        "tags.phase = 'final' AND tags.model_family = 'random_forest' "
        f"AND tags.workflow_version = '{WORKFLOW_VERSION}'"
    ),
    run_view_type=ViewType.ALL,
    max_results=100,
)
rf_diagnostic = next((run for run in rf_runs if run.info.status != "FINISHED"), None)
rf_diagnostic_rmse = (
    rf_diagnostic.data.metrics.get("holdout_rmse")
    if rf_diagnostic is not None else np.nan
)
docs = PROJECT_ROOT / "docs"
docs.mkdir(exist_ok=True)
runs.to_csv(docs / "corridas.csv", index=False)
pd.DataFrame([{"champion_registrado": champion_family, "challenger_registrado": challenger_family, "holdout_rmse_champion": expected_rmse, "holdout_rmse_challenger": trained[challenger_family]["holdout_rmse"], "random_forest_previous_attempt_status": rf_diagnostic.info.status if rf_diagnostic else "not-run", "random_forest_previous_attempt_rmse_diagnostic_only": rf_diagnostic_rmse, "random_forest_previous_attempt_run_id": rf_diagnostic.info.run_id if rf_diagnostic else "", "training_rows_full": train_rows, "holdout_rows_full": holdout_rows, "selection_sample_compounds": sample_metadata["compounds"], "selection_sample_sha256": sample_metadata["sha256"], "resource_adjusted_random_forest_cv_rmse": rf_resource_cv_rmse, "resource_adjusted_random_forest_cv_std": rf_resource_cv_std, "random_forest_max_samples": rf_resource_rows, **interval, **business_proxy}]).to_csv(docs / "seleccion_final.csv", index=False)
print(f"Exportadas {len(runs)} corridas de {WORKFLOW_VERSION} a docs/corridas.csv")

{'intervalo_95_pairwise': {'coverage': 0.95,
  'lower_error_quantile': -9.079015175118261,
  'upper_error_quantile': 9.003760573989693,
  'pair_error_radius': 9.079015175118261,
  'calibration_pairs': 52719},
 'proxy_sin_etiquetas_de_coelucion': {'comparison_pairs': 61742,
  'near_retention_proxy_pairs': 53643,
  'coelution_proxy_recall': 0.9658855768693027,
  'coelution_proxy_false_negative_rate': 0.03411442313069735,
  'potential_coelution_alert_rate': 0.9116322762463153,
  'pairwise_prediction_interval_coverage': 0.9570308703961646,
  'coelution_labels_are_proxy': 1}}

El recall/FNR es solo un proxy de cercanía temporal; no es una tasa real de coelución ni demuestra ahorro de corridas.


{'holdout_rmse_registrado': 2.328971943011868, 'reproduccion_exacta': True}


Exportadas 109 corridas de stratified-selection-full-final-v1 a docs/corridas.csv


In [4]:
from src.monitoring import feature_drift_report
from src.pipeline import clear_feature_caches

train_monitor_sample = X_train.sample(n=min(20_000, len(X_train)), random_state=42)
holdout_monitor_sample = X_test.sample(n=min(20_000, len(X_test)), random_state=42)
feature_transformer = registered_champion.named_steps["features"]
reference_features = feature_transformer.transform(train_monitor_sample)
current_features = feature_transformer.transform(holdout_monitor_sample)
drift_table = feature_drift_report(reference_features, current_features, bins=10)
drift_table.to_csv(docs / "feature_drift_holdout.csv", index=False)
display(drift_table)
print("Comparación offline de descriptores entre entrenamiento y holdout; no es monitoreo de producción.")
clear_feature_caches()
gc.collect()

,feature,psi,reference_n,current_n,reference_missing_fraction,current_missing_fraction
0,mol_h_donors,8.027573e-03,20000,20000,0.00000,0.00000
1,mol_h_acceptors,7.773127e-03,20000,20000,0.00000,0.00000
2,mol_tpsa,5.335118e-03,20000,20000,0.00000,0.00000
3,mol_weight,5.121069e-03,20000,20000,0.00000,0.00000
4,mol_logp,4.825100e-03,20000,20000,0.00000,0.00000
5,mol_ring_count,3.681896e-03,20000,20000,0.00000,0.00000
6,mol_fraction_csp3,3.019376e-03,20000,20000,0.00000,0.00000
7,mol_aromatic_rings,2.497676e-03,20000,20000,0.00000,0.00000
8,mol_rotatable_bonds,7.983464e-04,20000,20000,0.00000,0.00000
9,flow_ml_min,4.880945e-04,20000,20000,0.04355,0.04710


Comparación offline de descriptores entre entrenamiento y holdout; no es monitoreo de producción.


23

## Interpretación

La regla de alerta usa el intervalo de predicción del 95% de la diferencia entre dos tiempos de retención, calibrado con residuos fuera de pliegue. El recall/FNR se interpreta únicamente respecto al proxy de pares temporalmente cercanos. RepoRT no contiene resolución ni ancho de pico para confirmar coelución; el ahorro del 30% sigue siendo una hipótesis que requiere piloto y registros reales del laboratorio.

La celda final compara PSI y faltantes de descriptores entre muestras de entrenamiento y holdout sin usar el target. Es una demostración offline del indicador de deriva; en producción debe calcularse por lote nuevo y el RMSE/MAE solo cuando lleguen las retenciones reales.